# SASRec — Amazon Beauty
GPU accelerator: Settings → Accelerator → GPU T4 x2 (or P100)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os, subprocess, sys

REPO_DIR   = '/kaggle/working/SASRec.pytorch'
PYTHON_DIR = os.path.join(REPO_DIR, 'python')
DATA_DIR   = os.path.join(PYTHON_DIR, 'data')
RAW_CSV    = '/kaggle/working/ratings_Beauty.csv'
INTER_FILE = os.path.join(DATA_DIR, 'Beauty.txt')

# Clone
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/pmixer/SASRec.pytorch {REPO_DIR}
else:
    print('Repo already cloned')

In [ ]:
# Download dataset
import urllib.request
URL = 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/ratings_Beauty.csv'
if not os.path.isfile(RAW_CSV):
    print('Downloading...')
    urllib.request.urlretrieve(URL, RAW_CSV)
    print('Done')
else:
    print('Already downloaded')
print('Size:', os.path.getsize(RAW_CSV) // 1024, 'KB')

In [ ]:
# 5-core preprocess  →  Beauty.txt  (one uid iid pair per line)
import csv
from collections import defaultdict

if os.path.isfile(INTER_FILE):
    print('Beauty.txt already exists, skip.')
else:
    print('Loading CSV...')
    user_items = defaultdict(list)
    item_users = defaultdict(set)

    with open(RAW_CSV, 'r', encoding='utf-8') as f:
        for row in csv.reader(f):
            if len(row) < 4:
                continue
            u_raw, i_raw, _, ts_raw = row[0], row[1], row[2], row[3]
            try:
                ts = int(float(ts_raw))
            except ValueError:
                continue
            user_items[u_raw].append((ts, i_raw))
            item_users[i_raw].add(u_raw)

    print('Applying 5-core filter...')
    while True:
        changed = False
        bad_items = {i for i, us in item_users.items() if len(us) < 5}
        if bad_items:
            changed = True
            for u in list(user_items):
                user_items[u] = [(t, i) for t, i in user_items[u] if i not in bad_items]
                if not user_items[u]:
                    del user_items[u]
            item_users = defaultdict(set)
            for u, seq in user_items.items():
                for _, i in seq:
                    item_users[i].add(u)
        bad_users = {u for u, seq in user_items.items() if len(seq) < 5}
        if bad_users:
            changed = True
            for u in bad_users:
                for _, i in user_items[u]:
                    item_users[i].discard(u)
                del user_items[u]
        if not changed:
            break

    print(f'After 5-core: {len(user_items)} users, {len(item_users)} items')

    user2id = {u: idx+1 for idx, u in enumerate(sorted(user_items))}
    item2id = {i: idx+1 for idx, i in enumerate(sorted(item_users))}

    os.makedirs(DATA_DIR, exist_ok=True)
    with open(INTER_FILE, 'w') as out:
        for u_raw, seq in sorted(user_items.items(), key=lambda x: user2id[x[0]]):
            uid = user2id[u_raw]
            for ts, i_raw in sorted(seq, key=lambda x: x[0]):
                out.write(f'{uid} {item2id[i_raw]}\n')
    print('Beauty.txt written.')

In [ ]:
# Train
import subprocess, sys
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on {device}')

cmd = [
    sys.executable, 'main.py',
    '--dataset',      'Beauty',
    '--train_dir',    'default',
    '--maxlen',       '50',
    '--hidden_units', '64',
    '--num_blocks',   '2',
    '--num_heads',    '2',
    '--dropout_rate', '0.5',
    '--num_epochs',   '50',
    '--batch_size',   '256',
    '--device',       device,
]
result = subprocess.run(cmd, cwd=PYTHON_DIR)
print('Return code:', result.returncode)

In [ ]:
# Show saved checkpoints
model_dir = os.path.join(PYTHON_DIR, 'Beauty_default')
print(f'Checkpoints in {model_dir}:')
for f in sorted(os.listdir(model_dir)):
    fp = os.path.join(model_dir, f)
    print(f'  {f}  ({os.path.getsize(fp)//1024} KB)')